## 1.0  Reconstruction Geometries [For Developers]:

All Jupyter notebooks in this folder are for developers and maintainers. Ideally functions added into the tomobase library should be built with pytests. However, some libraries require manual validation or testing. That is the purpose of this folder. 

This particular notebook is used for testing that reconstruction geometries are behaving as intended.

### Table of Contents

### 1.1 Overview

The Tomobase library has 3 underlying APIs for reconstructions. Astra, Tomosipo, and OPTomo, Its worth noting all three APIs ultimately use the ASTRA-Toolbox for reconstructions. However, how they are maintained and used are quite different.

Whilst the ASTRA-Toolbox can be flexibly utilized for any arbitrary grometry or algorithm, he methodology behind this is not straightforward for pythonic applications. In practice, its better to provide Astra with preset algorithms, geometries and allow it to manage these under the cover. In addition, the fact that 

OPtomo is the recommended bindings for users, who want to use pythonic annotation and it is maintained and distributed with all ASTRA-Toolbox versions. This allows users to construct tomographic algorithms with similar notations to the mathematic operations. Additionally it is faster than raw ASTRA because the geometry setup and data transfer to bindings is not being repeated. 

The one challenge with Optomo, is that certain backends or algorithms are not openly supported in the API. E.g. 3D Forward Projection without the GPU. For this reason, to allow full API construction all Optomo operations are conducted in 2D. 

Tomosipo is a library that gives backend flexibility to the ASTRA toolbox when using python. Importantly, tomosipo, allows backend support for CPU (Numpy) and GPU usage in both 2D and 3D (using CUPY - the default support in this library, and Torch a pretty critical tool for machine learning applications).

Ideally, if tomosipo was better maintained this would be the default library. Unfortunetly, maintainence is a bit limited for this reason both Tomosipo and Optomo are wrapped with a Projector class which will use the kernal as specified by the user. 

```mermaid
graph TD
    A[Tomobase Projector]
    B[OPtomo]
    C[Tomosipo]
    D[Astra]

    A --> B
    A --> C
    B --> D
    C --> D
```
The raw ASTRA kernel exists solely for the purpose of checking that the orientation of each kernel is consistent with the default of raw ASTRA. The following cells are used for performing the nessary geometry tests.  

### 1.2 Set Up
For the purposes of visualization, this notebook uses the Jupyter Bindings

In [ ]:
import logging
import tomobase
tomobase.bootstrap(jupyter_enabled=True)

from tomobase import phantoms, procedures, data_classes, logger, proxy, GPUContext
from tomobase import jupyter

logger.setLevel(logging.DEBUG)
jupyter.display_log() 

### 1.1. The Astra Kernel

The raw ASTRA kernel exists solely for the purpose of checking that the orientation of each kernel is consistent with the default of raw ASTRA. The following cells are used for performing the nessessary geometry tests.  First test, Astra forward and back projection are consistent. 

Note: the raw Astra kernal does not support multimodal tomography such as EDX-EELS.

#### Tests:
- Validate: Approximate Orientation of Sinogram and Reprojection

In [ ]:
#Use this cell if evaluating other kernels, to avoid having to re-evaluate the ASTRA projection every time.
import numpy as np
import astra
import copy
from tomobase import procedures, phantoms, progress, registers
from tomobase.core.data_classes import tiltschemes, images
from ipywidgets import VBox, HBox

astra.clear()
angles = np.array(tiltschemes.GRS()[0:100])
vol = phantoms.get_nanocage()
sinogram_astra = procedures.astra_project(vol, angles, inplace=False)
sinogram_astra.sort("angles") 

In [4]:
import numpy as np
import astra
import copy
from tomobase import procedures, phantoms, progress, registers
from tomobase.core.data_classes import tiltschemes, images
from ipywidgets import VBox, HBox

astra.clear()
angles = np.array(tiltschemes.GRS()[0:100])
vol = phantoms.get_nanocage()
sinogram_astra = procedures.astra_project(vol, angles, inplace=False)
sinogram_astra.sort("angles")

volume_astra = procedures.astra_reconstruct(sinogram_astra,iterations=10, inplace=False)
sinogram_astra2 = procedures.astra_project(volume_astra, angles, inplace=False)
sinogram_astra2.sort("angles")

widget = jupyter.SliceGrid([sinogram_astra, sinogram_astra2], columns=2)
display(widget)


2026-05-05 16:04:11,092 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:11,097 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:11,106 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

2026-05-05 16:04:23,527 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:23,527 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:23,543 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:23,565 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Back projecting'), HBox(children=(IntProgress(value=0, max=307), Labe…

2026-05-05 16:35:13,767 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:35:13,767 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:35:13,781 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:35:13,789 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

2026-05-05 16:35:26,278 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:35:26,279 - DEBUG - Set context to GPUContext.NUMPY on device 0


SliceGrid(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('ny', ('n', 'y')),…

### 1.2. The Optomo Kernel

This section is for validating the geometry of the optomo module. Note, The exact implementation of GPU/CPU drivers and reconstruction algorithms is not validated here.
#### Tests:
- Validate: Projection Matches Astra
- Validate: Reprojection Matches Projection

In [6]:



import logging
import tomobase
tomobase.bootstrap(jupyter_enabled=True)

from tomobase import phantoms, procedures, data_classes, logger, proxy, GPUContext
from tomobase import jupyter

logger.setLevel(logging.DEBUG)
#jupyter.display_log() 

import numpy as np
import astra
import copy
from tomobase import procedures, phantoms, progress, registers
from tomobase.core.data_classes import tiltschemes, images
from ipywidgets import VBox, HBox

astra.clear()
angles = np.array(tiltschemes.GRS()[0:100])
vol = phantoms.get_nanocage()
sinogram_astra = procedures.astra_project(vol, angles, inplace=False)
sinogram_astra.sort("angles") 






sinogram_optomo = procedures.project(vol, angles, kernel="astra", use_3D=True, inplace=False)
sinogram_optomo.sort("angles")

volume_optomo = procedures.reconstruct_sirt(sinogram_optomo, kernel="astra", use_3D=True, iterations=100, weighted=False, inplace=False)
print(np.min(volume_optomo.values), np.max(volume_optomo.values))
#volume_optomo.interactive.info()
sinogram_optomo2 = procedures.project(volume_optomo, angles, kernel="astra", use_3D=True,  inplace=False)
sinogram_optomo2.sort("angles")

widget = jupyter.SliceGrid([sinogram_astra, sinogram_optomo, sinogram_optomo2], columns=3)
display(widget)

ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

ProgressBarWidget(children=(Label(value='Iterative Back Projection'), HBox(children=(IntProgress(value=0), Lab…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

...

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

ProgressBarWidget(children=(Label(value='Slice Wise Back Projection'), HBox(children=(IntProgress(value=0, max…

-0.3912456 1.3638659


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

SliceGrid(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('ny', ('n', 'y')),…

### 1.2. The Optomo Kernel

This section is for validating the geometry of the tomosipo module. Note, The exact implementation of GPU/CPU drivers and reconstruction algorithms is not validated here.

#### Tests:
- Validate: Projection Matches Astra [2D]
- Validate: Reprojection Matches Projection [2D]
- Validate: Projection Matches Astra [3D]
- Validate: Reprojection Matches Projection [3D]

In [3]:


import logging
import tomobase
tomobase.bootstrap(jupyter_enabled=True)

from tomobase import phantoms, procedures, data_classes, logger, proxy, GPUContext
from tomobase import jupyter

logger.setLevel(logging.DEBUG)
#jupyter.display_log() 

import numpy as np
import astra
import copy
from tomobase import procedures, phantoms, progress, registers
from tomobase.core.data_classes import tiltschemes, images
from ipywidgets import VBox, HBox

astra.clear()
angles = np.array(tiltschemes.GRS()[0:100])
vol = phantoms.get_nanocage()
sinogram_astra = procedures.astra_project(vol, angles, inplace=False)
sinogram_astra.sort("angles") 

vol.data.values = vol.data.values.transpose(2,1,0) # This would be z, x, y (3D) 2,0, 1
sinogram_tp = procedures.project(vol, angles, kernel="tomosipo", use_3D=True, inplace=False)
sinogram_tp.sort("angles")

#volume_tp = procedures.reconstruct_wbp(sinogram_tp, kernel="tomosipo", use_3D=False, inplace=False)
sinogram_tp2 = procedures.project(vol, angles, kernel="tomosipo", use_3D=False, inplace=False)
sinogram_tp2.sort("angles")

widget = jupyter.SliceGrid([sinogram_astra, sinogram_tp, sinogram_tp2], columns=3)
display(widget)

2026-05-05 16:03:48,064 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:03:48,069 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:03:48,078 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

2026-05-05 16:04:00,536 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:00,537 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:00,552 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:00,560 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=1), Lab…

sino slice shape (307, 100, 307) {}


2026-05-05 16:04:01,024 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:01,025 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:01,041 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:01,049 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

sino slice shape (100, 307) {'y': 0}
sino slice shape (100, 307) {'y': 1}
sino slice shape (100, 307) {'y': 2}
sino slice shape (100, 307) {'y': 3}
sino slice shape (100, 307) {'y': 4}
sino slice shape (100, 307) {'y': 5}
sino slice shape (100, 307) {'y': 6}
sino slice shape (100, 307) {'y': 7}
sino slice shape (100, 307) {'y': 8}
sino slice shape (100, 307) {'y': 9}
sino slice shape (100, 307) {'y': 10}
sino slice shape (100, 307) {'y': 11}
sino slice shape (100, 307) {'y': 12}
sino slice shape (100, 307) {'y': 13}
sino slice shape (100, 307) {'y': 14}
sino slice shape (100, 307) {'y': 15}
sino slice shape (100, 307) {'y': 16}
sino slice shape (100, 307) {'y': 17}
sino slice shape (100, 307) {'y': 18}
sino slice shape (100, 307) {'y': 19}
sino slice shape (100, 307) {'y': 20}
sino slice shape (100, 307) {'y': 21}
sino slice shape (100, 307) {'y': 22}
sino slice shape (100, 307) {'y': 23}
sino slice shape (100, 307) {'y': 24}
sino slice shape (100, 307) {'y': 25}
sino slice shape (100,

2026-05-05 16:04:04,410 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 16:04:04,411 - DEBUG - Set context to GPUContext.NUMPY on device 0


SliceGrid(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('ny', ('n', 'y')),…

In [1]:

import logging
import tomobase
tomobase.bootstrap(jupyter_enabled=True)

from tomobase import phantoms, procedures, data_classes, logger, proxy, GPUContext
from tomobase import jupyter

logger.setLevel(logging.DEBUG)
#jupyter.display_log() 

import numpy as np
import astra
import copy
from tomobase import procedures, phantoms, progress, registers
from tomobase.core.data_classes import tiltschemes, images
from ipywidgets import VBox, HBox

astra.clear()
angles = np.array(tiltschemes.GRS()[0:100])
vol = phantoms.get_nanocage()
sinogram_astra = procedures.astra_project(vol, angles, inplace=False)
sinogram_astra.sort("angles") 

#vol.data.values = vol.data.values.transpose(2,1,0) # This would be z, y, x (3D) 2,0, 1
sinogram_tp = procedures.project(vol, angles, kernel="tomosipo", use_3D=True, inplace=False)
sinogram_tp.sort("angles")

volume_tp = procedures.reconstruct_mlem(sinogram_tp, kernel="tomosipo", use_3D=True, inplace=False)
sinogram_tp2 = procedures.project(volume_tp, angles, kernel="tomosipo", use_3D=True, inplace=False)
sinogram_tp2.sort("angles")

widget = jupyter.SliceGrid([sinogram_astra, sinogram_tp, sinogram_tp2], columns=3)
display(widget)

type tester <class 'type'> <class 'type'> <class 'type'>
called qt


2026-05-05 23:28:34,720 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:34,726 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:34,733 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=307), L…

2026-05-05 23:28:47,150 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:47,151 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:47,165 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:47,174 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=1), Lab…

...

2026-05-05 23:28:51,469 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:51,470 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:51,484 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:51,493 - DEBUG - Set context to GPUContext.NUMPY on device 0


ProgressBarWidget(children=(Label(value='Forward projecting'), HBox(children=(IntProgress(value=0, max=1), Lab…

sino slice shape (307, 100, 307) {}


2026-05-05 23:28:51,673 - DEBUG - Set context to GPUContext.NUMPY on device 0
2026-05-05 23:28:51,674 - DEBUG - Set context to GPUContext.NUMPY on device 0


SliceGrid(children=(ImageSliceWidget(children=(ToggleButtons(description='View:', options=(('ny', ('n', 'y')),…